In [ ]:
# Here only mutual information based feature selection is used. No need to run this ..

import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

# ======================================================
# CONFIGURATION
# ======================================================
BASE_DIR = Path("ML-Features_v1")
MERGED_DIR = Path("Merged_Features")
MERGED_DIR.mkdir(exist_ok=True, parents=True)

TOP_FEATURES_REPORT = Path("Top2_Features_Per_Channel.csv")

# ======================================================
# HELPER FUNCTION
# ======================================================
def select_top_features(X, y, k=2):
    """
    Select top-k features using Mutual Information.
    Returns feature names and MI scores.
    """
    mi = mutual_info_classif(X, y, random_state=42, discrete_features=False)
    mi_df = pd.DataFrame({"feature": X.columns, "MI_score": mi})
    mi_df = mi_df.sort_values(by="MI_score", ascending=False)
    return mi_df.head(k)

# ======================================================
# MAIN PIPELINE
# ======================================================
summary_rows = []

print(f"🔍 Starting feature merging and selection for all channels in {BASE_DIR}...")

channel_folders = sorted([d for d in BASE_DIR.iterdir() if d.is_dir()])

for channel_dir in channel_folders:
    channel_name = channel_dir.name
    csv_files = sorted(channel_dir.glob("*.csv"))
    print(f"\n📁 Processing channel: {channel_name} ({len(csv_files)} files)")

    dfs = []
    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path)
            dfs.append(df)
        except Exception as e:
            print(f"❌ Error reading {csv_path}: {e}")

    if not dfs:
        print(f"⚠️ No data for channel {channel_name}")
        continue

    # Merge all subject CSVs
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_path = MERGED_DIR / f"{channel_name}_merged.csv"
    merged_df.to_csv(merged_path, index=False)
    print(f"✅ Merged saved: {merged_path} (rows={len(merged_df)}, cols={len(merged_df.columns)})")

    # Prepare for feature selection
    if "label" not in merged_df.columns:
        print(f"⚠️ Missing 'label' column in {channel_name}, skipping.")
        continue

    X = merged_df.drop(columns=["label", "subj_id", "window_idx", "pain_score"], errors="ignore")
    y = merged_df["label"].astype(str)

    # Encode target labels numerically
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Drop any constant columns
    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index
    X = X.drop(columns=constant_cols)
    if len(constant_cols) > 0:
        print(f"⚠️ Dropped {len(constant_cols)} constant features for {channel_name}")

    # Select top-2 features
    top_feats = select_top_features(X, y_encoded, k=2)

    for _, row in top_feats.iterrows():
        summary_rows.append({
            "channel": channel_name,
            "feature": row["feature"],
            "MI_score": round(row["MI_score"], 6)
        })

    print(f"⭐ Top 2 features for {channel_name}:")
    print(top_feats)

# Save summary
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(TOP_FEATURES_REPORT, index=False)
print(f"\n✅ Top features report saved → {TOP_FEATURES_REPORT}")
print(summary_df.head(10))


In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.utils import Bunch
import warnings
warnings.filterwarnings("ignore")

# ======================================================
# CONFIGURATION
# ======================================================
BASE_DIR = Path("ML-Features_v1")
MERGED_DIR = Path("Merged_Features_AllMethods")
MERGED_DIR.mkdir(exist_ok=True, parents=True)

SUMMARY_FILE = Path("Top_Features_AllMethods_Summary.csv")

# ======================================================
# HELPER FUNCTIONS
# ======================================================
def safe_fit(X, y, model):
    """Fit a model safely with feature scaling."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model.fit(X_scaled, y)
    return model

def feature_importances_random_forest(X, y):
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X, y)
    return rf.feature_importances_

def feature_importances_xgb(X, y):
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    )
    xgb.fit(X, y)
    return xgb.feature_importances_

def feature_importances_logreg(X, y):
    logreg = LogisticRegression(max_iter=2000, solver="liblinear")
    model = safe_fit(X, y, logreg)
    imp = np.abs(model.coef_).mean(axis=0)
    return imp

def feature_importances_rfe(X, y):
    logreg = LogisticRegression(max_iter=2000, solver="liblinear")
    selector = RFE(logreg, n_features_to_select=max(2, X.shape[1] // 3))
    model = safe_fit(X, y, selector)
    ranking = model.ranking_
    imp = 1 / ranking.astype(float)
    return imp

def normalize_scores(scores):
    """Normalize any numeric array to [0,1]."""
    arr = np.array(scores)
    if np.all(arr == 0):
        return np.zeros_like(arr)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-12)
    return arr


In [2]:


# ======================================================
# MAIN PIPELINE
# ======================================================
summary_rows = []
channel_folders = sorted([d for d in BASE_DIR.iterdir() if d.is_dir()])

print(f"🚀 Starting hybrid feature selection on {len(channel_folders)} channels")

for channel_dir in channel_folders:
    channel_name = channel_dir.name
    csv_files = sorted(channel_dir.glob("*.csv"))
    print(f"\n📁 Channel: {channel_name} ({len(csv_files)} files)")

    dfs = []
    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path)
            dfs.append(df)
        except Exception as e:
            print(f"❌ Error reading {csv_path}: {e}")

    if not dfs:
        continue

    merged_df = pd.concat(dfs, ignore_index=True)
    merged_path = MERGED_DIR / f"{channel_name}_merged.csv"
    merged_df.to_csv(merged_path, index=False)

    if "label" not in merged_df.columns:
        print(f"⚠️ Missing label in {channel_name}, skipping.")
        continue

    X = merged_df.drop(columns=["label", "subj_id", "window_idx", "pain_score"], errors="ignore")
    y = LabelEncoder().fit_transform(merged_df["label"].astype(str))

    # Drop constants
    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index
    X = X.drop(columns=constant_cols)
    if len(constant_cols):
        print(f"⚠️ Dropped {len(constant_cols)} constant cols in {channel_name}")

    # Compute scores for each method
    results = Bunch()

    print("→ Running filter methods (ANOVA, MI)...")
    results.anova = f_classif(X, y)[0]
    results.mi = mutual_info_classif(X, y, random_state=42)

    print("→ Running embedded methods (RF, XGB, LogReg)...")
    results.rf = feature_importances_random_forest(X, y)
    results.xgb = feature_importances_xgb(X, y)
    results.logreg = feature_importances_logreg(X, y)

    print("→ Running wrapper method (RFE)...")
    results.rfe = feature_importances_rfe(X, y)

    # Combine into a single importance matrix
    importance_df = pd.DataFrame({
        "feature": X.columns,
        "anova": normalize_scores(results.anova),
        "mi": normalize_scores(results.mi),
        "rf": normalize_scores(results.rf),
        "xgb": normalize_scores(results.xgb),
        "logreg": normalize_scores(results.logreg),
        "rfe": normalize_scores(results.rfe)
    })

    # Average across methods
    importance_df["avg_score"] = importance_df[["anova", "mi", "rf", "xgb", "logreg", "rfe"]].mean(axis=1)
    importance_df = importance_df.sort_values("avg_score", ascending=False)

    # Top features
    top_feats = importance_df.head(2)
    for _, row in top_feats.iterrows():
        summary_rows.append({
            "channel": channel_name,
            "feature": row["feature"],
            "avg_score": round(row["avg_score"], 4),
            "anova": round(row["anova"], 4),
            "mi": round(row["mi"], 4),
            "rf": round(row["rf"], 4),
            "xgb": round(row["xgb"], 4),
            "logreg": round(row["logreg"], 4),
            "rfe": round(row["rfe"], 4)
        })

    # Save per-channel feature ranking
    channel_rank_path = MERGED_DIR / f"{channel_name}_feature_ranking.csv"
    importance_df.to_csv(channel_rank_path, index=False)
    print(f"✅ Saved ranking for {channel_name}: {channel_rank_path}")



🚀 Starting hybrid feature selection on 24 channels

📁 Channel: AFz (34 files)
→ Running filter methods (ANOVA, MI)...
→ Running embedded methods (RF, XGB, LogReg)...
→ Running wrapper method (RFE)...
✅ Saved ranking for AFz: Merged_Features_AllMethods\AFz_feature_ranking.csv

📁 Channel: C3 (34 files)
→ Running filter methods (ANOVA, MI)...
→ Running embedded methods (RF, XGB, LogReg)...
→ Running wrapper method (RFE)...
✅ Saved ranking for C3: Merged_Features_AllMethods\C3_feature_ranking.csv

📁 Channel: C4 (34 files)
→ Running filter methods (ANOVA, MI)...
→ Running embedded methods (RF, XGB, LogReg)...
→ Running wrapper method (RFE)...
✅ Saved ranking for C4: Merged_Features_AllMethods\C4_feature_ranking.csv

📁 Channel: CPz (34 files)
→ Running filter methods (ANOVA, MI)...
→ Running embedded methods (RF, XGB, LogReg)...
→ Running wrapper method (RFE)...
✅ Saved ranking for CPz: Merged_Features_AllMethods\CPz_feature_ranking.csv

📁 Channel: Cz (34 files)
→ Running filter methods (ANO

In [3]:

# Global summary
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_FILE, index=False)

print("\n🎯 Hybrid feature selection complete!")
print(f"📂 Top features summary saved → {SUMMARY_FILE}")
print(summary_df.head(10))



🎯 Hybrid feature selection complete!
📂 Top features summary saved → Top_Features_AllMethods_Summary.csv
  channel                feature  avg_score   anova      mi      rf     xgb  \
0     AFz      signal_energy_AFz     0.7246  0.9976  0.7108  0.5794  1.0000   
1     AFz  wavelet_energy_L0_AFz     0.6499  1.0000  0.7131  0.6254  0.5010   
2      C3       signal_energy_C3     0.7616  0.9999  0.7605  0.7490  0.7977   
3      C3              energy_C3     0.7292  0.9999  0.7605  0.5956  0.7569   
4      C4   wavelet_energy_L0_C4     0.6869  1.0000  0.6723  0.8101  0.5279   
5      C4                 p10_C4     0.6533  0.0972  0.9870  0.7982  1.0000   
6     CPz  wavelet_energy_L0_CPz     0.6582  0.5825  0.7087  0.5173  0.8974   
7     CPz                rms_CPz     0.6351  0.5191  0.7087  0.5074  0.6676   
8      Cz    spectral_edge_90_Cz     0.6812  1.0000  0.2220  0.6081  1.0000   
9      Cz          higuchi_fd_Cz     0.5957  0.8733  0.2237  0.5534  0.5807   

   logreg  rfe  
0  0.059

In [ ]:
# Overall best features 


import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.utils import Bunch
import warnings
warnings.filterwarnings("ignore")

# ======================================================
# CONFIGURATION
# ======================================================
BASE_DIR = Path("ML-Features_v1")
GLOBAL_OUT_DIR = Path("Global_Feature_Selection")
GLOBAL_OUT_DIR.mkdir(exist_ok=True, parents=True)

GLOBAL_MERGED_FILE = GLOBAL_OUT_DIR / "Global_Feature_Matrix.csv"
GLOBAL_RANK_FILE = GLOBAL_OUT_DIR / "Global_Feature_Ranking.csv"
GLOBAL_TOP48_FILE = GLOBAL_OUT_DIR / "Top_48_Global_Features.csv"

# ======================================================
# HELPER FUNCTIONS
# ======================================================
def safe_fit(X, y, model):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model.fit(X_scaled, y)
    return model

def feature_importances_random_forest(X, y):
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X, y)
    return rf.feature_importances_

def feature_importances_xgb(X, y):
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="mlogloss"
    )
    xgb.fit(X, y)
    return xgb.feature_importances_

def feature_importances_logreg(X, y):
    logreg = LogisticRegression(max_iter=2000, solver="liblinear")
    model = safe_fit(X, y, logreg)
    imp = np.abs(model.coef_).mean(axis=0)
    return imp

def feature_importances_rfe(X, y):
    logreg = LogisticRegression(max_iter=2000, solver="liblinear")
    selector = RFE(logreg, n_features_to_select=max(5, X.shape[1] // 5))
    model = safe_fit(X, y, selector)
    ranking = model.ranking_
    imp = 1 / ranking.astype(float)
    return imp

def normalize_scores(scores):
    arr = np.array(scores)
    if np.all(arr == 0):
        return np.zeros_like(arr)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-12)
    return arr

# ======================================================
# STEP 1 — MERGE ALL CHANNELS BY SUBJECT
# ======================================================
print("🔍 Step 1: Merging all channel features per subject...")

channel_folders = sorted([d for d in BASE_DIR.iterdir() if d.is_dir()])
subject_ids = set()

# Collect all subject IDs
for channel_dir in channel_folders:
    for csv_file in channel_dir.glob("ID*_feature.csv"):
        subj = int(''.join([c for c in csv_file.stem if c.isdigit()]))
        subject_ids.add(subj)
subject_ids = sorted(list(subject_ids))
print(f"Found {len(subject_ids)} subjects across all channels.")

global_dfs = []

for subj in subject_ids:
    subj_dfs = []
    for channel_dir in channel_folders:
        channel_name = channel_dir.name
        file_path = channel_dir / f"ID{subj}_feature.csv"
        if not file_path.exists():
            continue
        df = pd.read_csv(file_path)
        keep_cols = [c for c in df.columns if c not in ["subj_id", "pain_score", "label", "window_idx"]]
        keep_cols = ["window_idx", "label"] + [c for c in df.columns if c not in ["subj_id", "pain_score", "label"]]
        subj_dfs.append(df[keep_cols])

    if not subj_dfs:
        continue

    # Merge all channels for this subject on window_idx + label
    merged_subj = subj_dfs[0]
    for dfc in subj_dfs[1:]:
        merged_subj = pd.merge(merged_subj, dfc, on=["window_idx", "label"], how="inner")

    merged_subj["subj_id"] = subj
    global_dfs.append(merged_subj)

# Combine all subjects into one DataFrame
global_df = pd.concat(global_dfs, ignore_index=True)
global_df.to_csv(GLOBAL_MERGED_FILE, index=False)
print(f"✅ Global merged feature matrix saved: {GLOBAL_MERGED_FILE}")
print(f"Shape: {global_df.shape}")

# ======================================================
# STEP 2 — FEATURE SELECTION ON GLOBAL DATA
# ======================================================
print("\n🔍 Step 2: Performing hybrid feature selection on global matrix...")

X = global_df.drop(columns=["label", "subj_id", "window_idx"], errors="ignore")
y = LabelEncoder().fit_transform(global_df["label"].astype(str))

# Drop constant features
nunique = X.nunique()
const_cols = nunique[nunique <= 1].index
if len(const_cols):
    print(f"⚠️ Dropping {len(const_cols)} constant features.")
    X = X.drop(columns=const_cols)

# Apply multiple methods
results = Bunch()

print("→ Running Filter methods (ANOVA, MI)...")
results.anova = f_classif(X, y)[0]
results.mi = mutual_info_classif(X, y, random_state=42)

print("→ Running Embedded methods (RF, XGB, LogReg)...")
results.rf = feature_importances_random_forest(X, y)
results.xgb = feature_importances_xgb(X, y)
results.logreg = feature_importances_logreg(X, y)

print("→ Running Wrapper method (RFE)...")
results.rfe = feature_importances_rfe(X, y)

# Combine and normalize
feature_rank_df = pd.DataFrame({
    "feature": X.columns,
    "anova": normalize_scores(results.anova),
    "mi": normalize_scores(results.mi),
    "rf": normalize_scores(results.rf),
    "xgb": normalize_scores(results.xgb),
    "logreg": normalize_scores(results.logreg),
    "rfe": normalize_scores(results.rfe)
})
feature_rank_df["avg_score"] = feature_rank_df[
    ["anova", "mi", "rf", "xgb", "logreg", "rfe"]
].mean(axis=1)

feature_rank_df = feature_rank_df.sort_values("avg_score", ascending=False)
feature_rank_df.to_csv(GLOBAL_RANK_FILE, index=False)
print(f"✅ Global feature ranking saved → {GLOBAL_RANK_FILE}")

# Select top 48
top48 = feature_rank_df.head(48)
top48.to_csv(GLOBAL_TOP48_FILE, index=False)
print(f"🏆 Top 48 global features saved → {GLOBAL_TOP48_FILE}")

print("\n🎯 Global feature selection complete!")
print(top48.head(10))
